# Customer Segmentation & Revenue Strategy
## Online Retail II - UK E-commerce analysis


**Business Context:**
-- A UK based e-commerce company allocated budget across
all customer segments equally. The CMO( Cheif marketing Officer )
needs a data-driven insights that can help to reallocate budget 
to high spending customers and re-engaging with at-risk ones.

**Key Questions:**
1. Who are our most valuable customers and what defines them ?
2. Which customers are at risk of churning ?
3. How should marketing budget be redistributed across segments ?
4. At what point do most customers stop purchasing ?


**Dataset** : Online Retail II (UCI ML Repository)
**Dataset_Source** : https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci
**Tools** : Python (Pandas, Numpy, Matlplotlib, Seaborn, Scipy, Tableau, Statistics)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12,6)
plt.rcParams['font.size'] = 12


In [4]:
#Loading data

df1 = pd.read_excel("D:\90_Days\Project-1\online_retail_II.xlsx",sheet_name = 'Year 2009-2010')

In [5]:
df2 = pd.read_excel("D:\90_Days\Project-1\online_retail_II.xlsx",sheet_name = 'Year 2010-2011')

In [6]:
df = pd.concat([df1,df2],ignore_index=True)
print(f"Total records: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

Total records: 1,067,371
Total columns: 8
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


## 1. Data Inspection

Before performing analysis step, I understand data columns and values,
examine raw data structure, data types, To find missing values, Anomalies

This step prevents from misassumptions.

In [7]:
print(df.head(10))
print("\n")
print(df.tail(10))
print("\n")
print(df.sample(10))
print("\n")
print(df.columns.tolist())
print("\n")

#Datatype
print(df.dtypes)

print(df.describe())
print("\n")
print(df.info(memory_usage='deep'))

  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
5  489434     22064           PINK DOUGHNUT TRINKET POT         24   
6  489434     21871                  SAVE THE PLANET MUG        24   
7  489434     21523   FANCY FONT HOME SWEET HOME DOORMAT        10   
8  489435     22350                            CAT BOWL         12   
9  489435     22349       DOG BOWL , CHASING BALL DESIGN        12   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2 2009-12-01 07:45:00   6.75      13085.0  United

In [8]:
# Column - by - Column inspection

for col in df.columns:
    print(f"\n{'='*50}")
    print("Column_Name:",col)
    print(f"Column_Type: {df[col].dtype}")
    print(f"Null value Count: {df[col].isnull().sum()}")
    print(f"Null Value Percentage: {df[col].isnull().mean()*100:.2f}%")
    print(f"Unique Values: {df[col].nunique()}")
    if df[col].dtype in ['object','Str']:
        print(f"Sample Values: {df[col].dropna().unique()[:5]}")
    elif df[col].dtype in ['int64','datetime64[ns]','float64']:
        print(f"Min : {df[col].min()}, Max : {df[col].max()}")


Column_Name: Invoice
Column_Type: object
Null value Count: 0
Null Value Percentage: 0.00%
Unique Values: 53628
Sample Values: [489434 489435 489436 489437 489438]

Column_Name: StockCode
Column_Type: object
Null value Count: 0
Null Value Percentage: 0.00%
Unique Values: 5305
Sample Values: [85048 '79323P' '79323W' 22041 21232]

Column_Name: Description
Column_Type: object
Null value Count: 4382
Null Value Percentage: 0.41%
Unique Values: 5698
Sample Values: ['15CM CHRISTMAS GLASS BALL 20 LIGHTS' 'PINK CHERRY LIGHTS'
 ' WHITE CHERRY LIGHTS' 'RECORD FRAME 7" SINGLE SIZE '
 'STRAWBERRY CERAMIC TRINKET BOX']

Column_Name: Quantity
Column_Type: int64
Null value Count: 0
Null Value Percentage: 0.00%
Unique Values: 1057
Min : -80995, Max : 80995

Column_Name: InvoiceDate
Column_Type: datetime64[ns]
Null value Count: 0
Null Value Percentage: 0.00%
Unique Values: 47635
Min : 2009-12-01 07:45:00, Max : 2011-12-09 12:50:00

Column_Name: Price
Column_Type: float64
Null value Count: 0
Null Value P

Column      | Type          | Null  | Issues Found
Invoice     | Object        |   0   | As cancelled orders Invoice start with 'C', They are valid but we need to ignore them when calculating totals
StockCode   | Object        |   0   | Some values contain administratvie codes, They are not 'errors' but should be ignored when performing product analysis
Description | Object        |  4382 | As Price is '0' for all Null Description values, keeping them may mislead in calculating revenue trends
Quantity    | Int64         |   0   | 22950 values in this column are below '0', Which is 2% of overall data, Removing this doesn't make noticeable impact 
Inovice_Date| datetime[ns]  |   0   | No issues found
Price       | float64       |   0   | 6207 Values are price less than '0', Ignoring this rows will lead to minute mis-assumptions when performing profit-trends analysis
Customer ID | float64       | 243007| Customer id with Null values should be removed, As it leads to wrong customer segmentation
Country     | Object        |   0   | There are 756 values are' Unspecified' in 'Country' attribute, These need to be removed, As they disturb in country-wise analysis

In [203]:
print(df.loc[df['Quantity']<1,'Quantity'].count())
df.sample(50)
df[df['Price']<=0].count()
print(sorted(df['Country'].unique()))
df[df['Country'] == 'Unspecified'].count()

22950
['Australia', 'Austria', 'Bahrain', 'Belgium', 'Bermuda', 'Brazil', 'Canada', 'Channel Islands', 'Cyprus', 'Czech Republic', 'Denmark', 'EIRE', 'European Community', 'Finland', 'France', 'Germany', 'Greece', 'Hong Kong', 'Iceland', 'Israel', 'Italy', 'Japan', 'Korea', 'Lebanon', 'Lithuania', 'Malta', 'Netherlands', 'Nigeria', 'Norway', 'Poland', 'Portugal', 'RSA', 'Saudi Arabia', 'Singapore', 'Spain', 'Sweden', 'Switzerland', 'Thailand', 'USA', 'United Arab Emirates', 'United Kingdom', 'Unspecified', 'West Indies']


Invoice        756
StockCode      756
Description    756
Quantity       756
InvoiceDate    756
Price          756
Customer ID    524
Country        756
dtype: int64

In [221]:
ddf = df.copy()

## 2.Data Cleaning

In [224]:
# Removing Null values in customer_id
ddf = ddf[ddf['Customer ID'].notnull()]
# Removing Null values in Description
ddf = ddf[ddf['Description'].notnull()]
# Removing Price values < 0
ddf = ddf[ddf['Price'] > 0]
# Removing countries with value 'Unspecified'
ddf = ddf[ddf['Country'] != 'Unspecified']

In [225]:
ddf.count()

Invoice        823769
StockCode      823769
Description    823769
Quantity       823769
InvoiceDate    823769
Price          823769
Customer ID    823769
Country        823769
dtype: int64

True
